# Adding Main Gentlemen Variables

Joins `main_gentlemen.csv` to the parish shapefile using the same 20 km proximity logic applied to the Percy/disgruntled-gentlemen variables in `jn_01`. For each role category in the CSV, a binary dummy is created indicating whether a parish centroid falls within 20 km of at least one gentleman in that category.

**Input:** `Data/Raw/CSV/main_gentlemen.csv`, `Data/Processed/northParishFlows.shp`  
**Output:** `Data/Processed/northParishFlows.shp` (updated in place)

## Loading

In [1]:
print("Loading packages and data...")
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

# Paths relative to project root
PROJECT_ROOT = Path.cwd().parent
RAW  = PROJECT_ROOT / 'Data' / 'Raw'
PROCESSED = PROJECT_ROOT / 'Data' / 'Processed'

# Input paths
GENTLEMEN_CSV  = RAW / 'CSV' / 'main_gentlemen.csv'
PARISH_SHP     = PROCESSED / 'northParishFlows.shp'

# Output path
OUTPUT_SHP     = PROCESSED / 'northParishFlows.shp'

gentlemen_df = pd.read_csv(GENTLEMEN_CSV)
parish_flows = gpd.read_file(PARISH_SHP)

print(f"Loaded {len(gentlemen_df)} gentlemen from main_gentlemen.csv")
print(f"Loaded {len(parish_flows)} parishes")
print(f"\nRole distribution in CSV:")
role_source_cols = ['Active_Loyalist', 'Reluctant_Loyalist', 'Neutral',
                    'Reluctant_Rebel', 'Rebel_Participant', 'Active_Rebel']
for col in role_source_cols:
    print(f"  {col}: {int(gentlemen_df[col].sum())}")

Loading packages and data...
Loaded 39 gentlemen from main_gentlemen.csv
Loaded 1755 parishes

Role distribution in CSV:
  Active_Loyalist: 18
  Reluctant_Loyalist: 3
  Neutral: 6
  Reluctant_Rebel: 6
  Rebel_Participant: 2
  Active_Rebel: 4


## Convert to GeoDataFrame and reproject to BNG

Coordinates in the CSV are WGS84 (EPSG:4326); the parish shapefile uses British National Grid (EPSG:27700). Reproject so that the 20 km buffer is in metres.

In [2]:
gentlemen_gdf = gpd.GeoDataFrame(
    gentlemen_df,
    geometry=gpd.points_from_xy(gentlemen_df['Longitude'], gentlemen_df['Latitude']),
    crs='EPSG:4326'
).to_crs('EPSG:27700')

print(f"GeoDataFrame CRS: {gentlemen_gdf.crs}")
print(f"Bounding box (BNG metres): {gentlemen_gdf.total_bounds}")

GeoDataFrame CRS: EPSG:27700
Bounding box (BNG metres): [343032.96322354 292240.34013049 523868.68717606 613581.53241191]


## Create 20 km proximity variables by role

For each variable, buffer the relevant gentleman points by 20 000 m, union the buffers, then flag any parish whose centroid falls within the result.

| Output column | Definition |
|---|---|
| `mg_any`     | Any gentleman in the dataset |
| `mg_rebel`   | Rebel_Participant **or** Active_Rebel |
| `mg_act_reb` | Active_Rebel only |
| `mg_part`    | Rebel_Participant only |
| `mg_loyal`   | Active_Loyalist **or** Reluctant_Loyalist |
| `mg_act_loy` | Active_Loyalist only |
| `mg_neutral` | Neutral |
| `mg_rel_reb` | Reluctant_Rebel only |

Note: shapefile column names are truncated to 10 characters on save — all names here are already within that limit.

In [3]:
BUFFER_M = 20_000  # 20 km in metres

# Maps output column name → list of source columns whose union defines the group
# (None = all gentlemen)
role_definitions = {
    'mg_any':     None,
    'mg_rebel':   ['Rebel_Participant', 'Active_Rebel'],
    'mg_act_reb': ['Active_Rebel'],
    'mg_part':    ['Rebel_Participant'],
    'mg_loyal':   ['Active_Loyalist', 'Reluctant_Loyalist'],
    'mg_act_loy': ['Active_Loyalist'],
    'mg_neutral': ['Neutral'],
    'mg_rel_reb': ['Reluctant_Rebel'],
}

for out_col, source_cols in role_definitions.items():
    if source_cols is None:
        subset = gentlemen_gdf
    else:
        mask = gentlemen_gdf[source_cols].any(axis=1)
        subset = gentlemen_gdf[mask]

    if len(subset) == 0:
        parish_flows[out_col] = 0
        print(f"{out_col}: 0 gentlemen → 0 parishes flagged")
    else:
        buffer = subset.geometry.buffer(BUFFER_M).union_all()
        parish_flows[out_col] = parish_flows.geometry.centroid.within(buffer).astype(int)
        n_gentlemen = len(subset)
        n_parishes  = int(parish_flows[out_col].sum())
        print(f"{out_col}: {n_gentlemen} gentlemen → {n_parishes} parishes flagged")

mg_any: 39 gentlemen → 907 parishes flagged
mg_rebel: 6 gentlemen → 256 parishes flagged
mg_act_reb: 4 gentlemen → 168 parishes flagged
mg_part: 2 gentlemen → 88 parishes flagged
mg_loyal: 21 gentlemen → 668 parishes flagged
mg_act_loy: 18 gentlemen → 617 parishes flagged
mg_neutral: 6 gentlemen → 233 parishes flagged
mg_rel_reb: 6 gentlemen → 255 parishes flagged


## Inverse-distance-weighted (IDW) versions

For each parish centroid and role group, sum a weight across all gentlemen in the group:

```
w(d) = 1          if d ≤ 10 km
w(d) = 10 / d_km  if d > 10 km
```

So a parish 20 km away contributes 0.5, 30 km → 0.33, etc. The IDW score is the sum of weights across all gentlemen in the group.

| Output column | Definition |
|---|---|
| `mg_any_w`   | Any gentleman |
| `mg_rebel_w` | Rebel_Participant **or** Active_Rebel |
| `mg_areb_w`  | Active_Rebel only |
| `mg_part_w`  | Rebel_Participant only |
| `mg_loyal_w` | Active_Loyalist **or** Reluctant_Loyalist |
| `mg_aloy_w`  | Active_Loyalist only |
| `mg_neut_w`  | Neutral |
| `mg_rreb_w`  | Reluctant_Rebel only |

In [ ]:
FLAT_RADIUS_M = 10_000  # 10 km flat zone; beyond this weight = 10_000 / d_m

# Parish centroid coordinates as numpy arrays (BNG metres)
centroids = parish_flows.geometry.centroid
cx = centroids.x.values
cy = centroids.y.values

# IDW column name → source columns (same groupings as binary variables above)
idw_definitions = {
    'mg_any_w':   None,
    'mg_rebel_w': ['Rebel_Participant', 'Active_Rebel'],
    'mg_areb_w':  ['Active_Rebel'],
    'mg_part_w':  ['Rebel_Participant'],
    'mg_loyal_w': ['Active_Loyalist', 'Reluctant_Loyalist'],
    'mg_aloy_w':  ['Active_Loyalist'],
    'mg_neut_w':  ['Neutral'],
    'mg_rreb_w':  ['Reluctant_Rebel'],
}

for out_col, source_cols in idw_definitions.items():
    if source_cols is None:
        subset = gentlemen_gdf
    else:
        mask = gentlemen_gdf[source_cols].any(axis=1)
        subset = gentlemen_gdf[mask]

    if len(subset) == 0:
        parish_flows[out_col] = 0.0
        print(f"{out_col}: 0 gentlemen → all zeros")
        continue

    gx = subset.geometry.x.values
    gy = subset.geometry.y.values

    # Pairwise distances: shape (n_parishes, n_gentlemen)
    dist_m = np.sqrt((cx[:, np.newaxis] - gx[np.newaxis, :]) ** 2
                     + (cy[:, np.newaxis] - gy[np.newaxis, :]) ** 2)

    # Weight: 1 inside flat zone, 10_000/d outside (avoids division-by-zero at d=0)
    weights = np.where(dist_m <= FLAT_RADIUS_M, 1.0, FLAT_RADIUS_M / dist_m)

    parish_flows[out_col] = weights.sum(axis=1)
    col = parish_flows[out_col]
    print(f"{out_col}: {len(subset)} gentlemen  "
          f"min={col.min():.3f}  mean={col.mean():.3f}  max={col.max():.3f}")

## Save updated shapefile

In [4]:
parish_flows.to_file(OUTPUT_SHP)

print(f"Updated shapefile saved to {OUTPUT_SHP}")
print(f"\nNew columns added:")
for out_col in role_definitions:
    print(f"  - {out_col}: {int(parish_flows[out_col].sum())} parishes = 1")

Updated shapefile saved to c:\Users\nicho\My Drive\20_Projects\NRP---New-Rebellion-Paper\Data\Processed\northParishFlows.shp

New columns added:
  - mg_any: 907 parishes = 1
  - mg_rebel: 256 parishes = 1
  - mg_act_reb: 168 parishes = 1
  - mg_part: 88 parishes = 1
  - mg_loyal: 668 parishes = 1
  - mg_act_loy: 617 parishes = 1
  - mg_neutral: 233 parishes = 1
  - mg_rel_reb: 255 parishes = 1
